# examples

In [7]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torchsummary import summary

## Get Device for Training

In [8]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


## Define the Class

In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # 相当于
        self.linear_relu_stack = nn.Sequential(#按顺序执行模块
            nn.Linear(28*28, 512),#（输入维度，隐藏层维度）
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [10]:
model = NeuralNetwork().to(device)
print(model)
# 直接看到每一层

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [12]:
summary(model, (1,28,28))
# 可以看到参数情况

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
              ReLU-3                  [-1, 512]               0
            Linear-4                  [-1, 512]         262,656
              ReLU-5                  [-1, 512]               0
            Linear-6                   [-1, 10]           5,130
Total params: 669,706
Trainable params: 669,706
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 2.55
Estimated Total Size (MB): 2.58
----------------------------------------------------------------


In [15]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
# 表示沿着第一个维度（通常是行）求取张量中每行最大值的索引
# pred_probab（1*10）
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6], device='cuda:0')


## Model Layers

In [17]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


### nn.Flatten

In [19]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())
# 只保留batch_size和一个维度

torch.Size([3, 784])

### nn.Linear

In [20]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


### nn.ReLU

In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")
# 负数全都变为0

### nn.Sequential

In [22]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

### nn.Softmax

In [23]:
softmax = nn.Softmax(dim=1)
# 对其中的某一维进行softmax
pred_probab = softmax(logits)

## Model Parameters

In [24]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0047, -0.0043, -0.0008,  ..., -0.0056, -0.0179, -0.0351],
        [ 0.0168, -0.0065, -0.0184,  ..., -0.0253, -0.0249, -0.0344]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0061,  0.0245], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0234, -0.0298, -0.0288,  ..., -0.0100, -0.0245,  0.0172],
        [ 0.0008,  0.0326, -0.0383,  ..., -0.0063, -0.0429,  0.0055]],
       device='cuda:0', grad_fn=<Sl

# nn.Module

## apply
apply(fn) 初始化参数 

In [25]:
@torch.no_grad()
def init_weights(m):
    print(m)
    if type(m) == nn.Linear:
        m.weight.fill_(1.0)
        print(m.weight)
net = nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 2))
net.apply(init_weights)

Linear(in_features=2, out_features=2, bias=True)
Parameter containing:
tensor([[1., 1.],
        [1., 1.]], requires_grad=True)
Linear(in_features=2, out_features=2, bias=True)
Parameter containing:
tensor([[1., 1.],
        [1., 1.]], requires_grad=True)
Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Linear(in_features=2, out_features=2, bias=True)
)


Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): Linear(in_features=2, out_features=2, bias=True)
)

## bfloat16()
可以把浮点数参数都改为这个类型
## buffers(recurse=True)
用于存储不需要进行梯度计算或更新的数据
模块的状态

## cpu()
## cuda(device=None)
Move all model parameters and buffers to the GPU.
## eval()
Set the module in evaluation mode.
This is equivalent with ```self.train(False)```
## train()

## save and load
### 保存模型的权重
#### load
```model.load_state_dict(torch.load(weight_path))```

#### save
```torch.save(model.state_dict(),weight_path)```

## named_parameters()
获取参数的名称和具体内容

In [30]:
for name, param in model.named_parameters():
    print(name)
    print(param.size())

linear_relu_stack.0.weight
torch.Size([512, 784])
linear_relu_stack.0.bias
torch.Size([512])
linear_relu_stack.2.weight
torch.Size([512, 512])
linear_relu_stack.2.bias
torch.Size([512])
linear_relu_stack.4.weight
torch.Size([10, 512])
linear_relu_stack.4.bias
torch.Size([10])


## requires_grad_(requires_grad=True)
是否需要梯度更新

## zero_grad()
只需要对优化器进行该操作

## register_buffer()
## register_module()
## register_parameter()
创建可学习参数

In [32]:
import torch
import torch.nn as nn

class GaussianModel(nn.Module):

    def __init__(self):
        super(GaussianModel, self).__init__()

        self.register_parameter('mean', nn.Parameter(torch.zeros(1),
                                                     requires_grad=True))
        # 'mean'这个参数会被自动的放入parameter的字典中
        
        self.pdf = torch.distributions.Normal(self.state_dict()['mean'],
                                              torch.tensor([1.0]))
        # 创建了一个正态分布对象self.pdf,其均值为可学习参数mean，标准差为1.0
    def forward(self, x):
        return -self.pdf.log_prob(x)

# model = GaussianModel()

GaussianModel()

## register_module()
## add_module()

## model.get_submodule()
## model.get_buffer()
## model.get_parameter()

## modules parameters buffers
 _modules
 _parameters

In [39]:
class Test(nn.Module):
    def __init__(self):
        super(Test, self).__init__()
        self.linear1 = torch.nn.Linear(2,3)
        self.linear2 = torch.nn.Linear(3,4)
        self.batch_norm = nn.BatchNorm2d(4)

test_module = Test()
# test_module.to(torch.double)
print(test_module._modules)
print(test_module._modules['linear1'])
print(test_module._modules['linear1'].weight)
print(test_module._modules['linear1'].weight.dtype)

OrderedDict([('linear1', Linear(in_features=2, out_features=3, bias=True)), ('linear2', Linear(in_features=3, out_features=4, bias=True)), ('batch_norm', BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True))])
Linear(in_features=2, out_features=3, bias=True)
Parameter containing:
tensor([[ 5.8878e-01,  4.5890e-01],
        [-3.8006e-01,  1.7946e-01],
        [ 1.7901e-01,  2.5374e-04]], dtype=torch.float64, requires_grad=True)
torch.float64


In [40]:
test_module._parameters

OrderedDict()

In [41]:
test_module._buffers
# 没有递归的逻辑

OrderedDict()

In [42]:
test_module.state_dict().keys()
# 就是一个字典啦

odict_keys(['linear1.weight', 'linear1.bias', 'linear2.weight', 'linear2.bias', 'batch_norm.weight', 'batch_norm.bias', 'batch_norm.running_mean', 'batch_norm.running_var', 'batch_norm.num_batches_tracked'])

In [ ]:
for p in test_module.parameters():
    print(p)

In [44]:
for p in test_module.buffers():
    print(p)

tensor([0., 0., 0., 0.], dtype=torch.float64)
tensor([1., 1., 1., 1.], dtype=torch.float64)
tensor(0)


In [ ]:
for p in test_module.named_parameters():
    print(p)

In [46]:
for p in test_module.named_children():
    print(p)

('linear1', Linear(in_features=2, out_features=3, bias=True))
('linear2', Linear(in_features=3, out_features=4, bias=True))
('batch_norm', BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True))


# Sequential()
如果传入的是字典就按有序字典导入
如果传入的直接是模块的话，就会按照有序数字进行命名